# Stage 3 — QSVM Meta-Labeling ★ HEADLINE

**The keystone result for the deck.** Train a quantum-kernel SVM to predict `P(primary alpha signal is correct)` using **orthogonal meta-features** (features the alpha model didn't see). Compare against classical RBF SVM with bootstrap CI on the AUC delta.

## What success looks like
- QSVM AUC > RBF AUC on test set
- Bootstrap 95% CI on (QSVM_AUC − RBF_AUC) **excludes zero** → statistically significant
- Filtering primary alpha by `meta_prob > 0.5` improves Sharpe

## Prereqs
- `02_alpha_models.ipynb` run → `alpha_mu.parquet` (primary column)
- `01_data_signals.ipynb` run → `meta_features.parquet` + `master_dataset.parquet`

In [12]:
import os, sys, time, warnings, json, pickle
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve
from sklearn.utils import resample
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

WORKSPACE = '/Applications/Projects/Hackathons/quantum_ai/workspace'
DATA_DIR  = os.path.join(WORKSPACE, 'data')
MODELS_DIR = os.path.join(WORKSPACE, 'models')

# ====== Configuration ======
N_QUBITS         = 4               # quantum kernel feature map dim
N_TRAIN_SAMPLES  = 500             # subsample (quantum kernel is O(N^2))
N_TEST_SAMPLES   = 200
BOOTSTRAP_N      = 500             # bootstrap reps for CI
FEATURE_MAP_REPS = 2               # ZZFeatureMap repetitions
SVC_C            = 1.0             # regularization

# Top-K meta-features by mutual info (we'll select dynamically)
TOP_K_FEATURES = N_QUBITS

CHECKS = {}
def check(name, condition, msg='', critical=True):
    status = 'PASS' if condition else ('FAIL' if critical else 'WARN')
    CHECKS[name] = (status, msg)
    icon = '[PASS]' if status == 'PASS' else ('[FAIL]' if status == 'FAIL' else '[WARN]')
    print(f'{icon} {name}: {msg}')
    return condition

print(f'Workspace: {WORKSPACE}')
print(f'QSVM config: {N_QUBITS} qubits, {FEATURE_MAP_REPS} feature-map reps, C={SVC_C}')
print(f'Sample caps: N_train={N_TRAIN_SAMPLES}, N_test={N_TEST_SAMPLES}, bootstrap={BOOTSTRAP_N}')

Workspace: /Applications/Projects/Hackathons/quantum_ai/workspace
QSVM config: 4 qubits, 2 feature-map reps, C=1.0
Sample caps: N_train=500, N_test=200, bootstrap=500


## 1. Load alpha predictions + meta-features + labels

In [13]:
# Alpha predictions (primary column = best model from Stage 2)
alpha_mu = pd.read_parquet(os.path.join(DATA_DIR, 'alpha_mu.parquet'))
print(f'alpha_mu: {alpha_mu.shape}, columns: {list(alpha_mu.columns)}')

# Meta features (orthogonal to alpha signals)
meta_long = pd.read_parquet(os.path.join(DATA_DIR, 'meta_features.parquet'))
# meta_features.parquet has MultiIndex columns (ticker, feature). Convert to long.
meta_long = meta_long.stack(level=0, dropna=False)
meta_long.index = meta_long.index.set_names(['timestamp', 'ticker'])
print(f'meta_features: {meta_long.shape}, columns: {list(meta_long.columns)}')

# Forward returns (the truth — was the alpha signal correct?)
fwd_5d = pd.read_parquet(os.path.join(DATA_DIR, 'forward_returns_5d.parquet'))
fwd_long = fwd_5d.stack(dropna=False).rename('fwd_return_5d')
fwd_long.index = fwd_long.index.set_names(['timestamp', 'ticker'])
print(f'fwd_returns: {len(fwd_long)} rows')

# Stage 2 primary metadata
with open(os.path.join(DATA_DIR, 'stage2_primary.json')) as f:
    primary_meta = json.load(f)
primary_name = primary_meta['primary_model']
print(f'\nPrimary alpha model (from Stage 2): {primary_name}  (IC={primary_meta["IC"]:.4f})')

check('alpha_mu loaded', 'primary' in alpha_mu.columns, f'primary col present')
check('meta_features loaded', len(meta_long) > 0, f'{len(meta_long):,} rows')
check('forward returns loaded', len(fwd_long) > 0, f'{len(fwd_long):,} rows')

alpha_mu: (12180, 15), columns: ['mom5d_baseline', 'xpyq', 'linear', 'ridge', 'random_forest', 'lightgbm', 'vqc', 'lstm', 'chronos', 'timesfm', 'timesfm_ft', 'kronos', 'kronos_ft', 'ensemble', 'primary']
meta_features: (37460, 7), columns: ['vol_regime', 'news_count_3d', 'news_disp_3d', 'vol_z_30d', 'dow', 'month_end', 'quarter_end']
fwd_returns: 37460 rows

Primary alpha model (from Stage 2): RandomForest  (IC=0.0916)
[PASS] alpha_mu loaded: primary col present
[PASS] meta_features loaded: 37,460 rows
[PASS] forward returns loaded: 37,460 rows


True

## 2. Build meta-label

Meta-label = **was the primary alpha signal directionally correct?**

- `1` = primary predicted up and actual went up (or both down)
- `0` = primary predicted up but actual went down (or vice versa)

This is binary classification — perfect for QSVM.

In [14]:
# Join primary alpha with forward returns
primary_alpha = alpha_mu['primary']
joined = pd.concat([primary_alpha.rename('alpha'), fwd_long.rename('fwd')], axis=1).dropna()
print(f'Joined alpha + fwd: {len(joined):,} rows')

# Meta-label: 1 if directionally correct
joined['correct'] = (np.sign(joined['alpha']) == np.sign(joined['fwd'])).astype(int)

# Also keep abs(alpha) as a feature (high-confidence predictions tend to be more accurate)
joined['alpha_strength'] = joined['alpha'].abs()

# Class balance check
pos_rate = joined['correct'].mean()
print(f'Meta-label balance: {pos_rate:.2%} correct  ({(joined["correct"] == 1).sum():,} positive, {(joined["correct"] == 0).sum():,} negative)')

check('Meta-label balanced (40-60%)', 0.40 < pos_rate < 0.65,
      f'{pos_rate:.2%} positive (close to majority class baseline)')

Joined alpha + fwd: 12,180 rows
Meta-label balance: 58.10% correct  (7,077 positive, 5,103 negative)
[PASS] Meta-label balanced (40-60%): 58.10% positive (close to majority class baseline)


np.True_

## 3. Build orthogonal feature matrix + train/test split

In [15]:
# Join meta-features
X_full = joined[['alpha_strength']].join(meta_long, how='inner').dropna()
y_full = joined.loc[X_full.index, 'correct']
print(f'Feature matrix: {X_full.shape}, label: {y_full.shape}')
print(f'Feature columns: {list(X_full.columns)}')

# Note: alpha_mu only contains predictions on the Stage 2 TEST window (2018-2020).
# So we split THAT window: earlier dates -> Stage 3 train, later dates -> Stage 3 test.
ts = X_full.index.get_level_values('timestamp')
unique_dates = sorted(ts.unique())
print(f'\nAvailable date range: {unique_dates[0].date()} -> {unique_dates[-1].date()} ({len(unique_dates)} days)')

# Use 60/40 chronological split — earlier 60% trains the meta-classifier,
# later 40% tests it (and notably includes COVID March 2020 if window spans into 2020)
split_pos = int(len(unique_dates) * 0.60)
S3_TRAIN_END  = unique_dates[split_pos - 1]
S3_TEST_START = unique_dates[split_pos]
print(f'Stage 3 train: <= {S3_TRAIN_END.date()}  ({split_pos} days)')
print(f'Stage 3 test:  >= {S3_TEST_START.date()}  ({len(unique_dates) - split_pos} days)')

train_mask = ts <= S3_TRAIN_END
test_mask  = ts >= S3_TEST_START

X_train_full = X_full[train_mask]
y_train_full = y_full[train_mask]
X_test_full  = X_full[test_mask]
y_test_full  = y_full[test_mask]

print(f'\nTrain: {len(X_train_full):,} rows  ({y_train_full.mean():.1%} positive)')
print(f'Test:  {len(X_test_full):,} rows  ({y_test_full.mean():.1%} positive)')

check('Train/test sufficient', len(X_train_full) > 1000 and len(X_test_full) > 500,
      f'train={len(X_train_full):,}, test={len(X_test_full):,}')


Feature matrix: (12180, 8), label: (12180,)
Feature columns: ['alpha_strength', 'vol_regime', 'news_count_3d', 'news_disp_3d', 'vol_z_30d', 'dow', 'month_end', 'quarter_end']

Available date range: 2018-01-02 -> 2020-06-03 (609 days)
Stage 3 train: <= 2019-06-14  (365 days)
Stage 3 test:  >= 2019-06-17  (244 days)

Train: 7,300 rows  (58.1% positive)
Test:  4,880 rows  (58.1% positive)
[PASS] Train/test sufficient: train=7,300, test=4,880


True

## 4. Feature selection — pick top-K most discriminative meta-features

QSVM is limited by qubit count. Pick the top-N_QUBITS features by mutual information.

In [16]:
from sklearn.feature_selection import mutual_info_classif

# Fill NaN for MI computation
X_train_filled = X_train_full.fillna(X_train_full.median())

mi_scores = mutual_info_classif(X_train_filled, y_train_full, random_state=42)
mi_df = pd.Series(mi_scores, index=X_train_full.columns).sort_values(ascending=False)
print('Mutual information per feature:')
print(mi_df.round(4).to_string())

TOP_FEATURES = mi_df.head(TOP_K_FEATURES).index.tolist()
print(f'\nSelected top {TOP_K_FEATURES} features for QSVM: {TOP_FEATURES}')

X_train_sel = X_train_full[TOP_FEATURES].fillna(X_train_full[TOP_FEATURES].median()).values
X_test_sel  = X_test_full[TOP_FEATURES].fillna(X_train_full[TOP_FEATURES].median()).values
y_train     = y_train_full.values
y_test      = y_test_full.values

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sel)
X_test_scaled  = scaler.transform(X_test_sel)

# Subsample for tractability (quantum kernel is O(N^2))
np.random.seed(42)
if len(X_train_scaled) > N_TRAIN_SAMPLES:
    train_idx = np.random.choice(len(X_train_scaled), N_TRAIN_SAMPLES, replace=False)
    X_train_q = X_train_scaled[train_idx]
    y_train_q = y_train[train_idx]
else:
    X_train_q, y_train_q = X_train_scaled, y_train
if len(X_test_scaled) > N_TEST_SAMPLES:
    test_idx = np.random.choice(len(X_test_scaled), N_TEST_SAMPLES, replace=False)
    X_test_q = X_test_scaled[test_idx]
    y_test_q = y_test[test_idx]
else:
    X_test_q, y_test_q = X_test_scaled, y_test
    test_idx = np.arange(len(X_test_scaled))

print(f'\nSubsampled for QSVM: train={X_train_q.shape}, test={X_test_q.shape}')
print(f'Class balance — train: {y_train_q.mean():.1%}, test: {y_test_q.mean():.1%}')

Mutual information per feature:
news_count_3d     0.0106
vol_z_30d         0.0063
news_disp_3d      0.0060
vol_regime        0.0042
alpha_strength    0.0033
dow               0.0000
month_end         0.0000
quarter_end       0.0000

Selected top 4 features for QSVM: ['news_count_3d', 'vol_z_30d', 'news_disp_3d', 'vol_regime']

Subsampled for QSVM: train=(500, 4), test=(200, 4)
Class balance — train: 60.4%, test: 56.5%


## 5. QSVM training ★ KEYSTONE

Quantum kernel via ZZFeatureMap. Computes pairwise quantum-state overlaps to build N×N kernel matrix, feeds into sklearn's SVC.

In [17]:
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel

print(f'Building ZZFeatureMap ({N_QUBITS} qubits, reps={FEATURE_MAP_REPS}, entanglement=linear)')
feature_map = ZZFeatureMap(feature_dimension=N_QUBITS, reps=FEATURE_MAP_REPS, entanglement='linear')
quantum_kernel = FidelityQuantumKernel(feature_map=feature_map)

print(f'Computing K_train ({N_TRAIN_SAMPLES}x{N_TRAIN_SAMPLES} = {N_TRAIN_SAMPLES**2:,} kernel evals)...')
t0 = time.time()
K_train = quantum_kernel.evaluate(x_vec=X_train_q)
print(f'  done in {time.time()-t0:.0f}s — K_train shape={K_train.shape}, mean={K_train.mean():.4f}')

print(f'Computing K_test ({N_TEST_SAMPLES}x{N_TRAIN_SAMPLES} = {N_TEST_SAMPLES * N_TRAIN_SAMPLES:,} kernel evals)...')
t0 = time.time()
K_test = quantum_kernel.evaluate(x_vec=X_test_q, y_vec=X_train_q)
print(f'  done in {time.time()-t0:.0f}s — K_test shape={K_test.shape}')

# Save the kernel matrix (Q-Atlas will reuse it in Stage 5.5!)
np.save(os.path.join(DATA_DIR, 'K_train_qsvm.npy'), K_train)
np.save(os.path.join(DATA_DIR, 'K_test_qsvm.npy'),  K_test)
np.save(os.path.join(DATA_DIR, 'X_train_qsvm.npy'), X_train_q)
np.save(os.path.join(DATA_DIR, 'X_test_qsvm.npy'),  X_test_q)
with open(os.path.join(MODELS_DIR, 'qsvm_meta.json'), 'w') as f:
    json.dump({'features': TOP_FEATURES, 'n_qubits': N_QUBITS,
               'reps': FEATURE_MAP_REPS, 'entanglement': 'linear'}, f)
print(f'\nSaved K_train, K_test, X_*, and qsvm metadata for Stage 5.5 (Q-Atlas)')

# Train SVM on quantum kernel
print(f'\nTraining QSVM (SVC with precomputed quantum kernel, C={SVC_C})...')
t0 = time.time()
qsvm = SVC(kernel='precomputed', C=SVC_C, probability=True, random_state=42)
qsvm.fit(K_train, y_train_q)
print(f'  trained in {time.time()-t0:.1f}s')

# Predict probabilities on test set
qsvm_proba = qsvm.predict_proba(K_test)[:, 1]
qsvm_pred  = qsvm.predict(K_test)

qsvm_auc = roc_auc_score(y_test_q, qsvm_proba)
qsvm_acc = accuracy_score(y_test_q, qsvm_pred)
print(f'\nQSVM test scores:')
print(f'  AUC = {qsvm_auc:.4f}')
print(f'  Acc = {qsvm_acc:.4f}')

check('QSVM trained', qsvm_auc > 0.5, f'AUC={qsvm_auc:.4f}')

ModuleNotFoundError: No module named 'qiskit_machine_learning'

## 6. Classical RBF-SVM baseline

Same data, same SVM, **classical RBF kernel**. The deck-headline comparison.

In [ ]:
print('Training classical RBF-SVM baseline...')
t0 = time.time()
rbf = SVC(kernel='rbf', C=SVC_C, gamma='scale', probability=True, random_state=42)
rbf.fit(X_train_q, y_train_q)
print(f'  trained in {time.time()-t0:.1f}s')

rbf_proba = rbf.predict_proba(X_test_q)[:, 1]
rbf_pred  = rbf.predict(X_test_q)

rbf_auc = roc_auc_score(y_test_q, rbf_proba)
rbf_acc = accuracy_score(y_test_q, rbf_pred)
print(f'\nRBF-SVM test scores:')
print(f'  AUC = {rbf_auc:.4f}')
print(f'  Acc = {rbf_acc:.4f}')

# Headline delta
auc_delta = qsvm_auc - rbf_auc
acc_delta = qsvm_acc - rbf_acc
print(f'\n=== HEADLINE: QSVM vs RBF ===')
print(f'  Δ AUC = {auc_delta:+.4f}  ({"QSVM wins" if auc_delta > 0 else "RBF wins"})')
print(f'  Δ Acc = {acc_delta:+.4f}')

check('QSVM beats RBF', auc_delta > 0, f'Δ AUC = {auc_delta:+.4f}', critical=False)

## 7. Bootstrap 95% CI on AUC delta ★ THE RIGOR

If the lower bound of the CI excludes zero → QSVM is **statistically significantly** better than RBF.

In [ ]:
print(f'Bootstrap test set ({BOOTSTRAP_N} resamples)...')
t0 = time.time()
deltas = []
for i in range(BOOTSTRAP_N):
    idx = resample(range(len(y_test_q)), random_state=i)
    if len(np.unique(y_test_q[idx])) < 2:
        continue
    q = roc_auc_score(y_test_q[idx], qsvm_proba[idx])
    r = roc_auc_score(y_test_q[idx], rbf_proba[idx])
    deltas.append(q - r)
deltas = np.array(deltas)
ci = np.percentile(deltas, [2.5, 97.5])
print(f'  done in {time.time()-t0:.0f}s')

print(f'\nΔ AUC (QSVM - RBF):')
print(f'  point estimate: {auc_delta:+.4f}')
print(f'  95% CI:         [{ci[0]:+.4f}, {ci[1]:+.4f}]')
print(f'  bootstrap mean: {deltas.mean():+.4f}')

ci_excludes_zero = ci[0] > 0
print(f'\n=== STATISTICAL SIGNIFICANCE ===')
if ci_excludes_zero:
    print(f'  ★ Lower 95% CI bound {ci[0]:+.4f} > 0  →  QSVM significantly better (p < 0.05)')
elif ci[1] < 0:
    print(f'  ✗ Upper 95% CI bound {ci[1]:+.4f} < 0  →  RBF significantly better')
else:
    print(f'  − CI [{ci[0]:+.4f}, {ci[1]:+.4f}] straddles 0 — no significant difference')

# Save for deck
with open(os.path.join(DATA_DIR, 'stage3_headline.json'), 'w') as f:
    json.dump({
        'qsvm_auc': float(qsvm_auc),
        'rbf_auc':  float(rbf_auc),
        'auc_delta': float(auc_delta),
        'auc_delta_ci_lo': float(ci[0]),
        'auc_delta_ci_hi': float(ci[1]),
        'auc_delta_bootstrap_mean': float(deltas.mean()),
        'auc_delta_bootstrap_std':  float(deltas.std()),
        'significant':  bool(ci_excludes_zero),
        'n_train': N_TRAIN_SAMPLES,
        'n_test':  N_TEST_SAMPLES,
        'features': TOP_FEATURES,
    }, f, indent=2)

check('Bootstrap CI computed', not np.isnan(ci).any(), f'CI=[{ci[0]:+.4f}, {ci[1]:+.4f}]')

## 8. Visualize — ROC curves + bootstrap distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Stage 3 — QSVM Meta-Labeling vs Classical RBF Baseline', fontweight='bold')

# ROC curves
ax = axes[0]
fpr_q, tpr_q, _ = roc_curve(y_test_q, qsvm_proba)
fpr_r, tpr_r, _ = roc_curve(y_test_q, rbf_proba)
ax.plot(fpr_q, tpr_q, label=f'QSVM (AUC={qsvm_auc:.3f})', linewidth=2, color='#4caf50')
ax.plot(fpr_r, tpr_r, label=f'RBF  (AUC={rbf_auc:.3f})', linewidth=2, color='#2196f3')
ax.plot([0,1], [0,1], 'k--', alpha=0.3, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC: QSVM vs RBF on meta-labeling task')
ax.legend()
ax.grid(alpha=0.3)

# Bootstrap distribution
ax = axes[1]
ax.hist(deltas, bins=40, color='steelblue', alpha=0.7, edgecolor='black')
ax.axvline(0, color='red', linestyle='--', label='No difference')
ax.axvline(ci[0], color='black', linestyle=':', label=f'95% CI: [{ci[0]:+.3f}, {ci[1]:+.3f}]')
ax.axvline(ci[1], color='black', linestyle=':')
ax.axvline(auc_delta, color='green', linewidth=2, label=f'Point estimate: {auc_delta:+.4f}')
ax.set_xlabel('Δ AUC (QSVM − RBF)')
ax.set_ylabel('Bootstrap frequency')
ax.set_title(f'Bootstrap distribution ({BOOTSTRAP_N} resamples)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'stage3_qsvm_vs_rbf.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved chart → data/stage3_qsvm_vs_rbf.png')

## 9. Apply meta-prob filter to primary alpha — does filtering improve Sharpe?

Use QSVM's `meta_prob` to filter trades. Keep only trades where `meta_prob > threshold`. Compare filtered vs unfiltered Sharpe.

In [ ]:
# Apply QSVM to ALL test samples (not just subsampled) — need predictions per (date, ticker)
print('Generating QSVM meta-probabilities on full test set...')
X_test_full_sel = X_test_full[TOP_FEATURES].fillna(X_train_full[TOP_FEATURES].median()).values
X_test_full_scaled = scaler.transform(X_test_full_sel)

# Quantum kernel on full test set (this is the expensive part — but we cap it)
max_full = min(len(X_test_full_scaled), 2000)   # cap for speed
full_idx = np.random.RandomState(42).choice(len(X_test_full_scaled), max_full, replace=False) if len(X_test_full_scaled) > max_full else np.arange(len(X_test_full_scaled))
X_full_subset = X_test_full_scaled[full_idx]

print(f'  computing K({max_full} x {N_TRAIN_SAMPLES})...')
t0 = time.time()
K_full = quantum_kernel.evaluate(x_vec=X_full_subset, y_vec=X_train_q)
print(f'  done in {time.time()-t0:.0f}s')

meta_proba_full = qsvm.predict_proba(K_full)[:, 1]

# Build meta_prob DataFrame indexed by (date, ticker)
full_test_idx_full = X_test_full.iloc[full_idx].index
meta_prob = pd.Series(meta_proba_full, index=full_test_idx_full, name='meta_prob')

# Compute Sharpe with vs without filter
fwd_test = fwd_long.reindex(meta_prob.index)
alpha_test = alpha_mu['primary'].reindex(meta_prob.index)

def long_short_sharpe_df(alpha, fwd, top_n=5, horizon=5):
    d = pd.concat([alpha.rename('a'), fwd.rename('y')], axis=1).dropna().reset_index()
    daily = []
    for _, g in d.groupby('timestamp'):
        if len(g) < 2 * top_n:
            continue
        g_sorted = g.sort_values('a')
        daily.append(g_sorted.tail(top_n)['y'].mean() - g_sorted.head(top_n)['y'].mean())
    if not daily:
        return np.nan
    r = pd.Series(daily)
    return r.mean() / (r.std() + 1e-9) * np.sqrt(252 / horizon)

sharpe_unfilt = long_short_sharpe_df(alpha_test, fwd_test)

# Threshold sweep
thresholds = [0.40, 0.45, 0.50, 0.55, 0.60]
print(f'\n=== Sharpe under meta-prob filtering ===')
print(f'Unfiltered (all trades): Sharpe = {sharpe_unfilt:.3f}')
print(f'{"threshold":>10s} {"n_kept":>10s}  {"sharpe":>8s}  {"delta":>8s}')
print('-' * 50)
best_thresh, best_sharpe = 0.5, sharpe_unfilt
for thresh in thresholds:
    mask = meta_prob > thresh
    n_kept = mask.sum()
    alpha_filt = alpha_test.where(mask)
    sharpe = long_short_sharpe_df(alpha_filt, fwd_test)
    delta = sharpe - sharpe_unfilt
    if sharpe > best_sharpe:
        best_sharpe = sharpe
        best_thresh = thresh
    print(f'{thresh:>10.2f} {n_kept:>10,}  {sharpe:>8.3f}  {delta:>+8.3f}')

print(f'\nBest filter: threshold={best_thresh}, Sharpe={best_sharpe:.3f}  (Δ vs unfiltered = {best_sharpe - sharpe_unfilt:+.3f})')

check('Meta-labeling improves Sharpe', best_sharpe > sharpe_unfilt,
      f'best Sharpe {best_sharpe:.3f} vs unfiltered {sharpe_unfilt:.3f}', critical=False)

## 10. Save meta_prob for Stage 4 (QAOA portfolio)

In [ ]:
# Save meta_prob as parquet — Stage 4 will use this to filter the QAOA universe
meta_prob_df = meta_prob.reset_index().rename(columns={'timestamp': 'timestamp', 'ticker': 'ticker'})
meta_prob_df.to_parquet(os.path.join(DATA_DIR, 'meta_prob.parquet'))
print(f'Saved meta_prob.parquet: {meta_prob_df.shape}')
print(f'  → {DATA_DIR}/meta_prob.parquet')

# Save the QSVM SVC model + scaler for inference
with open(os.path.join(MODELS_DIR, 'qsvm_svc.pkl'), 'wb') as f:
    pickle.dump(qsvm, f)
with open(os.path.join(MODELS_DIR, 'qsvm_rbf_baseline.pkl'), 'wb') as f:
    pickle.dump(rbf, f)
with open(os.path.join(MODELS_DIR, 'meta_scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)
print(f'Saved QSVM + RBF + scaler -> {MODELS_DIR}/')

# Also save best threshold
with open(os.path.join(DATA_DIR, 'stage3_summary.json'), 'w') as f:
    json.dump({
        'qsvm_auc': float(qsvm_auc),
        'rbf_auc':  float(rbf_auc),
        'auc_delta': float(auc_delta),
        'auc_delta_ci': [float(ci[0]), float(ci[1])],
        'significant': bool(ci_excludes_zero),
        'sharpe_unfiltered': float(sharpe_unfilt),
        'sharpe_filtered':   float(best_sharpe),
        'best_threshold':    float(best_thresh),
        'top_features': TOP_FEATURES,
        'primary_model_from_stage2': primary_name,
    }, f, indent=2)

print(f'\n=== STAGE 3 SUMMARY ===')
print(f'QSVM AUC:        {qsvm_auc:.4f}')
print(f'RBF AUC:         {rbf_auc:.4f}')
print(f'Δ AUC:           {auc_delta:+.4f}  95%CI [{ci[0]:+.4f}, {ci[1]:+.4f}]')
print(f'Significant:     {ci_excludes_zero}')
print(f'Sharpe unfilt:   {sharpe_unfilt:.3f}')
print(f'Sharpe filtered: {best_sharpe:.3f}  (threshold={best_thresh})')
print(f'Sharpe delta:    {best_sharpe - sharpe_unfilt:+.3f}')

## 11. Stage 4 readiness summary

In [ ]:
print('=' * 72)
print(' STAGE 3 READINESS SUMMARY')
print('=' * 72)

passed = [n for n, (s, _) in CHECKS.items() if s == 'PASS']
failed = [n for n, (s, _) in CHECKS.items() if s == 'FAIL']
warned = [n for n, (s, _) in CHECKS.items() if s == 'WARN']

print(f'\nChecks: {len(passed)} pass, {len(failed)} fail, {len(warned)} warn')
for name, (status, msg) in CHECKS.items():
    icon = '[PASS]' if status == 'PASS' else ('[FAIL]' if status == 'FAIL' else '[WARN]')
    print(f'  {icon} {name}: {msg}')

print()
print('=' * 72)
if failed:
    print(f' OVERALL: RED — {len(failed)} critical fail(s)')
else:
    if ci_excludes_zero:
        print(f' OVERALL: GREEN — QSVM significantly beats RBF (CI excludes 0)')
        print(f'   ★ HEADLINE: QSVM AUC {qsvm_auc:.4f} vs RBF {rbf_auc:.4f} (Δ {auc_delta:+.4f})')
    elif auc_delta > 0:
        print(f' OVERALL: YELLOW — QSVM beats RBF point-wise but not statistically significant')
        print(f'   Try: more training samples, different feature map, different C value')
    else:
        print(f' OVERALL: ORANGE — RBF baseline beat QSVM on this task')
        print(f'   Try: more meta-features, different ZZFeatureMap reps, expanded training')

print()
print(' Outputs for Stage 4 (QAOA portfolio):')
print('   - data/meta_prob.parquet  (filter universe before QAOA)')
print('   - data/K_train_qsvm.npy   (Q-Atlas reuses this kernel matrix in Stage 5.5)')
print('   - data/stage3_summary.json')
print(' Saved models for inference:')
print('   - workspace/models/qsvm_svc.pkl + qsvm_rbf_baseline.pkl + meta_scaler.pkl')
print('=' * 72)

with open(os.path.join(DATA_DIR, 'stage3_checks.json'), 'w') as f:
    json.dump({n: {'status': s, 'msg': m} for n, (s, m) in CHECKS.items()}, f, indent=2)